# Propcode NSW DCP Scraper
Downloads NSW DCP entries from `app.propcode.com.au` as **the original PDF** (intercepted from the site's Adobe DC viewer) plus an HTML transcript extracted from those PDFs.

PropCode renders every DCP through Adobe's PDF web viewer — the page itself is just an empty `<div id="adobe-dc-view">` shell, so we can't scrape HTML directly. Each DCP is split across part URLs of the form:

```
/library/document/<state>/<slug>/file/0
/library/document/<state>/<slug>/file/1
/library/document/<state>/<slug>/file/2
...
```

This notebook brute-force iterates `/file/0`, `/file/1`, … in the browser, intercepts the PDF the Adobe SDK fetches for each part, downloads it via the authenticated session, then merges the parts into one canonical PDF and extracts an HTML transcript using `pymupdf`.

**Workflow:**
1. **Cell 2** — install packages (Playwright + pymupdf) + Playwright browsers
2. **Cell 3** — config (output folder, URLs)
3. **Cell 4** — one-time login: opens a browser window, you log in manually, session is saved to `session.json`
4. **Cell 5** — for each DCP in `nsw-dcp/DCPs.csv`: enumerate every part, save each as `<slug>-fileN.pdf`, combine into `<slug>.pdf`, extract `<slug>.html`
5. **Cell 6** — summary

> Run Cell 4 once. After that you can re-run Cell 5 any time without logging in again.

In [1]:
# Cell 2 — Install dependencies + Playwright browsers
import subprocess, sys

r1 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "playwright", "pymupdf"],
    capture_output=True, text=True
)
print(r1.stdout[-1500:] if len(r1.stdout) > 1500 else r1.stdout)

# Install the Playwright Chromium browser (only needed once)
r2 = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True, text=True
)
print(r2.stdout[-1000:] if len(r2.stdout) > 1000 else r2.stdout)

print("✓ Packages and browsers ready")

(from pyee<14,>=13->playwright) (4.9.0)
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB 165.2 kB/s eta 0:01:57
   ---------------------------------------- 0.0/19.2 MB 281.8 kB/s eta 0:01:09
   ---------------------------------------- 0.1/19.2 MB 590.8 kB/s eta 0:00:33
    --------------------------------------- 0.5/19.2 MB 2.2 MB/s eta 0:00:09
   --- ------------------------------------ 1.5/19.2 MB 6.0 MB/s eta 0:00:03
   ------- -------------------------------- 3.5/19.2 MB 11.7 MB/s eta 0:00:02
   --------- ------------------------------ 4.8/19.2 MB 13.9 MB/s eta 0:00:02
   -------------- ------------------------- 7.1/19.2 MB 18.2 MB/s eta 0:00:01
   -------------------- ------------------- 9.7/19.2 MB 22.1 MB/s eta 0:00:01
   ------------------------- -------------- 12.2/19.2 MB 46.7 MB/s eta 0:00:01
   ------------------------------ ----

In [2]:
# Cell 3 — Config
from pathlib import Path

BASE_URL     = "https://app.propcode.com.au"
SESSION_FILE = Path("nsw-dcp-session.json")   # saved after login
OUTPUT_DIR   = Path("nsw-dcp")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Session file : {SESSION_FILE.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Session file : C:\Luma\nsw-planning-library\public\EPI\nsw-dcp-session.json
Output folder: C:\Luma\nsw-planning-library\public\EPI\nsw-dcp


In [3]:

# Cell 4 — One-time login  (run once, then skip)
# Opens YOUR installed Chrome with automation flags hidden so Google OAuth works.
# Session is saved automatically after you're redirected back to the library.

import asyncio, sys
from pathlib import Path
from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

# Fallback definitions in case Cell 3 hasn't been run
try:
    BASE_URL
except NameError:
    BASE_URL = "https://app.propcode.com.au"
try:
    SESSION_FILE
except NameError:
    SESSION_FILE = Path("nsw-dcp-session.json")

def _login_worker():
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        asyncio.set_event_loop(asyncio.ProactorEventLoop())

    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=False,
            channel="chrome",
            args=["--disable-blink-features=AutomationControlled"],
            ignore_default_args=["--enable-automation"],
        )
        ctx = browser.new_context()
        page = ctx.new_page()
        page.goto(f"{BASE_URL}/library/nsw")

        # Step 1 — wait for the SPA to redirect us away from the library to the login page
        try:
            page.wait_for_url(lambda url: "/library/" not in url, timeout=15_000)
            print("▶ Redirected to login — complete the OAuth flow in the Chrome window...")
        except Exception:
            # If no redirect happened, we may already be on a login page or still loading
            print("▶ Chrome opened — complete the login in the Chrome window...")

        # Step 2 — wait up to 5 min for the post-auth redirect back to the library
        page.wait_for_url(f"{BASE_URL}/library/**", timeout=300_000)
        print("  Login detected — waiting for the app to fully load...")

        # Step 3 — give the React SPA time to finish initialising and store auth tokens
        try:
            page.wait_for_load_state("networkidle", timeout=30_000)
        except Exception:
            pass  # networkidle timeout is non-fatal; carry on
        page.wait_for_timeout(2_000)  # small extra buffer for localStorage writes

        ctx.storage_state(path=str(SESSION_FILE))
        browser.close()
        print(f"✓ Session saved → {SESSION_FILE}")

with ThreadPoolExecutor(max_workers=1) as ex:
    ex.submit(_login_worker).result()


▶ Chrome opened — complete the login in the Chrome window...
  Login detected — waiting for the app to fully load...
✓ Session saved → nsw-dcp-session.json


In [4]:

# Cell 5 — Bulk download every part of every DCP (real PDFs + HTML transcript)
#
# PropCode renders every DCP through Adobe's PDF web viewer — the page itself
# is just an empty <div id="adobe-dc-view"></div> shell, and each DCP's parts
# live at predictable URLs:
#       /library/document/<state>/<slug>/file/0
#       /library/document/<state>/<slug>/file/1
#       /library/document/<state>/<slug>/file/2 ...
#
# This cell:
#   1. Brute-force iterates /file/0, /file/1, … for each DCP.
#   2. For each part page, the Adobe SDK fetches a real PDF over the network —
#      we intercept that response and download the PDF via the authenticated
#      request context.
#   3. Stops once we hit several consecutive misses (no PDF / 404 / dupe).
#   4. Combines the parts into a single canonical <slug>.pdf and extracts an
#      HTML transcript via pymupdf.

import asyncio, csv, re, sys
from pathlib import Path
from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

try:
    import fitz  # pymupdf
except ImportError:
    fitz = None

# Fallback definitions in case Cell 3 hasn't been run
try:
    SESSION_FILE
except NameError:
    SESSION_FILE = Path("nsw-dcp-session.json")
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path("nsw-dcp")
    OUTPUT_DIR.mkdir(exist_ok=True)

# Tunables
MAX_FILES_PER_DCP = 80     # safety upper bound on /file/N enumeration
MAX_CONSEC_MISSES = 5      # stop after this many indices in a row return no PDF
PDF_WAIT_MS       = 15_000 # how long to wait for the Adobe SDK to fetch a PDF

# ── Load URLs from DCPs.csv (URL column) ──────────────────────────────────
CSV_FILE = OUTPUT_DIR / "DCPs.csv"
with CSV_FILE.open(encoding="utf-8") as f:
    DCP_URLS = [row["URL"].strip() for row in csv.DictReader(f) if row.get("URL", "").strip()]

print(f"Loaded {len(DCP_URLS)} URLs from {CSV_FILE}")
if fitz is None:
    print("⚠ pymupdf not installed — re-run Cell 2. PDFs will still download but HTML extraction will be skipped.")

# ──────────────────────────────────────────────────────────────────────────

def _slug_from_url(url: str) -> str:
    """Derive a safe filename slug from any DCP URL variant."""
    s = url.rstrip("/")
    s = re.sub(r"/file/\d+(/html)?/?$", "", s)
    s = re.sub(r"/html/?$", "", s)
    return s.rstrip("/").split("/")[-1][:80] or "unknown"


def _doc_base(url: str) -> str:
    """Strip /file/N and /html suffixes to get the canonical document URL."""
    s = url.rstrip("/")
    s = re.sub(r"/file/\d+(/html)?/?$", "", s)
    s = re.sub(r"/html/?$", "", s)
    return s


def _pdf_key(url: str) -> str:
    """Dedupe key — URL path without query string (signed-URL params change)."""
    return url.split("?", 1)[0]


def _make_pdf_handler(captured: list):
    """Playwright response handler that appends PDF URLs to `captured`."""
    def handler(response):
        try:
            ct  = (response.headers.get("content-type") or "").lower()
            url = response.url
            url_path = url.split("?", 1)[0].lower()
            is_pdf = ("pdf" in ct) or url_path.endswith(".pdf")
            if response.status == 200 and is_pdf and url not in captured:
                captured.append(url)
        except Exception:
            pass
    return handler


def _navigate_and_wait_for_pdf(page, url, captured) -> None:
    """
    Navigate to `url`, wait for the Adobe SDK to fetch its PDF.
    Mutates `captured` (cleared first).
    """
    captured.clear()
    try:
        page.goto(url, wait_until="domcontentloaded", timeout=60_000)
    except Exception as e:
        print(f"          ⚠ goto: {e}")
        return
    # Poll until we see at least one PDF response (or timeout)
    waited = 0
    while not captured and waited < PDF_WAIT_MS:
        page.wait_for_timeout(500)
        waited += 500
    # Brief settle to catch any additional PDF requests for the same part
    if captured:
        page.wait_for_timeout(1_500)


def _scrape_all_parts(page, ctx, base_url: str, slug: str,
                      output_dir: Path, captured: list) -> list:
    """Iterate /file/0, /file/1, … and download every unique PDF."""
    doc_base = _doc_base(base_url)
    pdf_paths: list = []
    seen_keys: set  = set()
    consecutive_misses = 0

    for n in range(MAX_FILES_PER_DCP):
        if consecutive_misses >= MAX_CONSEC_MISSES:
            break

        part_url = f"{doc_base}/file/{n}"
        _navigate_and_wait_for_pdf(page, part_url, captured)

        if not captured:
            consecutive_misses += 1
            print(f"        → file/{n:<3d} ✗ no PDF  (miss {consecutive_misses}/{MAX_CONSEC_MISSES})")
            continue

        # We got *some* PDF response — that means /file/N exists, so reset misses
        consecutive_misses = 0
        new_urls = [u for u in captured if _pdf_key(u) not in seen_keys]

        if not new_urls:
            print(f"        → file/{n:<3d} ⊙ duplicate of earlier part")
            continue

        print(f"        → file/{n:<3d}")
        for k, pdf_url in enumerate(new_urls):
            suffix = f"-{k+1}" if len(new_urls) > 1 else ""
            out = output_dir / f"{slug}-file{n}{suffix}.pdf"
            try:
                resp = ctx.request.get(pdf_url, timeout=120_000)
                if resp.ok:
                    body = resp.body()
                    out.write_bytes(body)
                    seen_keys.add(_pdf_key(pdf_url))
                    pdf_paths.append(out)
                    print(f"          ✓ {out.name} ({len(body):,} bytes)")
                else:
                    print(f"          ✗ HTTP {resp.status} for {pdf_url[:80]}")
            except Exception as e:
                print(f"          ✗ download failed: {e}")

    return pdf_paths


def _combine_pdfs(paths: list, out_path: Path) -> None:
    """Merge multiple PDFs into one using pymupdf (or copy if single)."""
    if fitz is None or len(paths) == 1:
        out_path.write_bytes(paths[0].read_bytes())
        return
    merged = fitz.open()
    try:
        for p in paths:
            with fitz.open(str(p)) as src:
                merged.insert_pdf(src)
        merged.save(str(out_path))
    finally:
        merged.close()


def _pdf_to_html(pdf_path: Path, title: str, source_url: str):
    """Extract HTML from a PDF using pymupdf."""
    if fitz is None:
        return None
    body_parts = []
    try:
        with fitz.open(str(pdf_path)) as doc:
            for i in range(len(doc)):
                body_parts.append(
                    f'<section class="pdf-page" data-page="{i+1}">\n'
                    + doc[i].get_text("html")
                    + "\n</section>"
                )
    except Exception as e:
        return f"<!-- pymupdf error: {e} -->"
    body = "\n".join(body_parts)
    return (
        '<!DOCTYPE html>\n<html lang="en">\n<head>\n'
        '  <meta charset="UTF-8">\n'
        f'  <meta name="source" content="{source_url}">\n'
        f'  <title>{title}</title>\n'
        '</head>\n<body>\n'
        f"{body}\n"
        '</body>\n</html>'
    )


def _download_worker(urls):
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        asyncio.set_event_loop(asyncio.ProactorEventLoop())

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        ctx = browser.new_context(storage_state=str(SESSION_FILE))
        page = ctx.new_page()

        # One shared PDF capture list, cleared between navigations
        captured: list = []
        page.on("response", _make_pdf_handler(captured))

        results = []
        for i, url in enumerate(urls, 1):
            slug     = _slug_from_url(url)
            html_out = OUTPUT_DIR / f"{slug}.html"
            pdf_out  = OUTPUT_DIR / f"{slug}.pdf"

            print(f"\n[{i:3d}/{len(urls)}] {slug}")
            try:
                # Quick visit to grab the document title
                title = slug
                try:
                    page.goto(url, wait_until="domcontentloaded", timeout=60_000)
                    page.wait_for_timeout(2_000)
                    t = page.title()
                    if t:
                        title = t
                except Exception:
                    pass

                # Brute-force enumerate parts and download each PDF
                part_paths = _scrape_all_parts(page, ctx, url, slug, OUTPUT_DIR, captured)

                if not part_paths:
                    raise RuntimeError("no PDFs captured — check session validity / site layout")

                # Combine into a single canonical PDF
                _combine_pdfs(part_paths, pdf_out)
                print(f"        ✓ {pdf_out.name} ({pdf_out.stat().st_size:,} bytes, {len(part_paths)} part(s))")

                # Extract HTML transcript from the canonical PDF
                html_doc = _pdf_to_html(pdf_out, title, url)
                html_chars = 0
                if html_doc:
                    html_out.write_text(html_doc, encoding="utf-8")
                    html_chars = len(html_doc)
                    print(f"        ✓ {html_out.name} ({html_chars:,} chars)")

                results.append({
                    "slug": slug,
                    "url": url,
                    "parts": len(part_paths),
                    "pdf_bytes": pdf_out.stat().st_size,
                    "html_chars": html_chars,
                    "status": "ok",
                })
            except Exception as e:
                print(f"        ✗ {e}")
                results.append({
                    "slug": slug, "url": url, "parts": 0,
                    "pdf_bytes": 0, "html_chars": 0, "status": str(e),
                })

        ctx.close()
        browser.close()
        return results


if not SESSION_FILE.exists():
    raise RuntimeError("Session file not found — run Cell 4 first to log in.")

with ThreadPoolExecutor(max_workers=1) as ex:
    results = ex.submit(_download_worker, DCP_URLS).result()


Loaded 5 URLs from nsw-dcp\DCPs.csv

[  1/5] randwick-comprehensive-dcp-2013
        → file/0  
          ✓ randwick-comprehensive-dcp-2013-file0.pdf (89,478 bytes)
        → file/1  
          ✓ randwick-comprehensive-dcp-2013-file1.pdf (5,832,110 bytes)
        → file/2  
          ✓ randwick-comprehensive-dcp-2013-file2.pdf (4,817,898 bytes)
        → file/3  
          ✓ randwick-comprehensive-dcp-2013-file3.pdf (3,230,453 bytes)
        → file/4  
          ✓ randwick-comprehensive-dcp-2013-file4.pdf (1,533,595 bytes)
        → file/5  
          ✓ randwick-comprehensive-dcp-2013-file5.pdf (2,174,232 bytes)
        → file/6  
          ✓ randwick-comprehensive-dcp-2013-file6.pdf (85,831 bytes)
        → file/7  
          ✓ randwick-comprehensive-dcp-2013-file7.pdf (597,234 bytes)
        → file/8  
          ✓ randwick-comprehensive-dcp-2013-file8.pdf (6,333,348 bytes)
        → file/9  
          ✓ randwick-comprehensive-dcp-2013-file9.pdf (1,672,611 bytes)
        → file/10 
  

In [5]:
# Cell 6 — Summary
ok     = [r for r in results if r["status"] == "ok"]
failed = [r for r in results if r["status"] != "ok"]

total_chars = sum(r["html_chars"] for r in ok)
total_pdf   = sum(r["pdf_bytes"]  for r in ok)
total_parts = sum(r["parts"]      for r in ok)

print(f"✓ Downloaded : {len(ok):3d} documents  ({total_parts} total parts)")
print(f"✗ Failed     : {len(failed):3d}")
print(f"Total PDF    : {total_pdf:,} bytes        ({total_pdf/1e6:.1f} MB)")
print(f"Total HTML   : {total_chars:,} characters  ({total_chars/1e6:.1f} MB)")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

if failed:
    print("\nFailed entries:")
    for r in failed:
        print(f"  ✗ {r['slug']:60s}  {r['status'][:80]}")

✓ Downloaded :   5 documents  (142 total parts)
✗ Failed     :   0
Total PDF    : 384,855,675 bytes        (384.9 MB)
Total HTML   : 624,373,332 characters  (624.4 MB)
Output folder: C:\Luma\nsw-planning-library\public\EPI\nsw-dcp
